# Calculating derived outputs

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/03-derived-outputs` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Beyond compartment sizes, models usually need flow rates, cumulative counts,
aggregates, and simple functions of those series. summer2 collected these via
`request_output_for_*`. In summer4 the same ideas are a `SavePlan` plus
queryable `Trace` methods: `FlowMass`, `Compartments`, `.select`, `.total`,
`.incidence`, `.cumulative`, and ordinary arithmetic on traces.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    ExitFlow,
    FlowMass,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

PARAMS = {"beta": 2.0}


def build_sir():
    m = FlowModel(pmap)
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    m.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection(
                "infection",
                infectious=state["I"],
                group_by=mixing.prop,
                mixing=mixing,
                kind="frequency",
                contact_rate=Param("beta"),
            ),
        )
    )
    m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    m.add_flow(ExitFlow("infection_death", state["I"], 0.05))
    return m


def y0_sir():
    y0 = jnp.zeros(pmap.size)
    y0 = y0.at[pmap.select(state["S"])].set(990.0)
    y0 = y0.at[pmap.select(state["I"])].set(10.0)
    return PropertyData.wrap(pmap, y0)


TS = jnp.linspace(0.0, 20.0, 201)


## Compartment trajectories

A plain SIR with infection deaths — the baseline for the queries below.


In [ ]:
m = build_sir()
cm = m.compile()
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=TS)
res = cm.run(PARAMS, y0_sir(), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
i_peak = float(np.max(np.asarray(res["comp"].select(state["I"]).values.data)))
assert i_peak > 10.0
res["comp"].to_pandas().plot(
    title="SIR compartments", labels={"index": "time (days)", "value": "people"}
)


## Flow outputs

Save instantaneous flow mass with `FlowMass`, then plot. `.incidence()` turns
rates into counts per save interval when you need person-time totals.


In [ ]:
m = build_sir()
plan = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "deaths": SaveRequest(FlowMass(flow="infection_death")),
    },
    ts=TS,
)
res = m.compile().run(PARAMS, y0_sir(), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
deaths = res["deaths"].total()
assert float(np.max(np.asarray(deaths.values))) > 0.0
deaths.to_pandas().plot(
    title="Infection-death flow rate",
    labels={"index": "time (days)", "value": "people / day"},
)


## Cumulative outputs

Integrate flow rates to interval counts (`.incidence()`), then `.cumulative()`
for a running total — the summer4 analogue of `request_cumulative_output`.


In [ ]:
deaths_cum = res["deaths"].incidence().cumulative().total()
final = float(np.asarray(deaths_cum.values)[-1])
assert final > 0.0
deaths_cum.to_pandas().plot(
    title="Cumulative infection deaths",
    labels={"index": "time (days)", "value": "people"},
)


## Compartment outputs

Select and sum compartments after the run — equivalent to
`request_output_for_compartments`.


In [ ]:
uninfected = res["comp"].select(state["S"] | state["R"]).total()
assert float(np.asarray(uninfected.values)[0]) == 990.0
uninfected.to_pandas().plot(
    title="Uninfected (S + R)",
    labels={"index": "time (days)", "value": "people"},
)


## Aggregate outputs

Combine several saved series with ordinary Trace arithmetic (summer2's
`request_aggregate_output`).


In [ ]:
m = build_sir()
plan = SavePlan(
    requests={
        "deaths": SaveRequest(FlowMass(flow="infection_death")),
        "recoveries": SaveRequest(FlowMass(flow="recovery")),
    },
    ts=TS,
)
res = m.compile().run(PARAMS, y0_sir(), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
deaths_c = res["deaths"].incidence().cumulative().total()
recov_c = res["recoveries"].incidence().cumulative().total()
ts = np.asarray(deaths_c.times.values)
combined = pd.Series(
    np.asarray(deaths_c.values).ravel() + np.asarray(recov_c.values).ravel(),
    index=ts,
    name="dead_or_recovered",
)
assert float(combined.iloc[-1]) > 0.0
combined.plot(
    title="Cumulative deaths + recoveries",
    labels={"index": "time (days)", "value": "people"},
)


## Function outputs

Prevalence is infectious / total. Traces do not overload arithmetic operators —
convert with `np.asarray` (or pandas) and combine on the host.


In [ ]:
m = build_sir()
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=TS)
res = m.compile().run(PARAMS, y0_sir(), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
i = np.asarray(res["comp"].select(state["I"]).total().values).ravel()
n = np.asarray(res["comp"].total().values).ravel()
ts = np.asarray(res["comp"].times.values)
prevalence = pd.Series(i / n, index=ts, name="prevalence")
assert 0.0 < float(prevalence.max()) < 1.0
prevalence.plot(
    title="Infectious prevalence",
    labels={"index": "time (days)", "value": "fraction"},
)


## Under `jax.jit`

Compile once, then differentiate through a parametric contact rate. Plotly /
`to_pandas` stay host-side after the run.


In [ ]:
cm = build_sir().compile()
y0 = y0_sir()
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=TS)


def loss(beta):
    res = cm.run({"beta": beta}, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
    return jnp.sum(jnp.asarray(res["comp"].select(state["I"]).values.data))


jitted = jax.jit(loss)
val = jitted(jnp.asarray(2.0))
grad = jax.grad(loss)(jnp.asarray(2.0))
assert jnp.isfinite(val) and jnp.isfinite(grad)
np.testing.assert_allclose(val, loss(jnp.asarray(2.0)), rtol=1e-4)
print(f"loss={float(val):.4g}, grad={float(grad):.4g}")


## Summary

| summer2 | summer4 |
|---|---|
| `request_output_for_flow` | `SaveRequest(FlowMass(...))` |
| `request_output_for_compartments` | `SaveRequest(Compartments())` + `.select` |
| `request_cumulative_output` | `.incidence().cumulative()` (or `.cumulative()` on rates) |
| `request_aggregate_output` | Host-side sum of saved traces |
| `request_function_output` | Host-side numpy/pandas on saved traces |
